### 模型训练流程

1. **数据准备**  
   加载并预处理训练数据。

2. **模型构建**  
   定义神经网络结构。

3. **定义损失函数与优化器**  
   选择合适的损失函数（如交叉熵）和优化器（如 Adam）。

4. **进入训练循环**  
   重复以下步骤直至训练完成：

   a. **迭代训练**  
      - 在训练集上进行前向传播、计算损失、反向传播和参数更新。

   b. **模型评估**  
      - 在验证集上评估当前模型性能（如准确率、损失等）。

   c. **判断是否更优？**  
      - 如果当前模型性能优于历史最佳：
        - **保存模型**（checkpoint）
      - 否则：
        - 继续下一轮训练

# 使用平均池化，即类似词袋的方式进行训练

In [ ]:
import os
import glob
import re
from collections import Counter
import numpy as np
def load_20newsgroups_manual(data_home, subset="train", categories=None):
    base = os.path.join(data_home, "20news-bydate", f"20news-bydate-{subset}")
    all_files = []
    all_targets = []
    target_names = categories if categories else sorted(os.listdir(base))
    
    for i, cat in enumerate(target_names):
        cat_path = os.path.join(base, cat)
        if not os.path.isdir(cat_path):
            continue
        for filepath in glob.glob(os.path.join(cat_path, "*")):
            with open(filepath, "rb") as f:
                content = f.read()
            # 尝试解码（原始数据是 latin-1 / iso-8859-1）
            try:
                text = content.decode("utf-8")
            except UnicodeDecodeError:
                text = content.decode("latin-1")
            all_files.append(text)
            all_targets.append(i)
    
    return {
        "data": all_files,
        "target": all_targets,
        "target_names": target_names
    }

# 使用
data_folder = "d:\\deeplearning\\scikit_learn_data"
categories = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

train_data_raw = load_20newsgroups_manual(data_folder, subset="train", categories=categories)
test_data_raw = load_20newsgroups_manual(data_folder, subset="test", categories=categories)


def basic_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9(),.!?\\'`]", " ", text)
    text = re.sub(r"([,.!?\\'`])", r" \\1 ", text)
    tokens = text.strip().split()
    return tokens

# 计算每篇文档的词元数量
train_text_lengths = [len(basic_tokenize(text)) for text in train_data_raw["data"]]

word_counts = Counter()
for text in train_data_raw["data"]:
    word_counts.update(basic_tokenize(text))

class Tokenizer:
    def __init__(self,vocab):
        self.vocab = vocab
        self.token_to_id = {token: idx for idx, token in enumerate(vocab)}
    
    @staticmethod
    def _tokenized_text(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9(),.!?\\'`]", " ", text)
        text = re.sub(r"([,.!?\\'`])", r" \\1 ", text)
        return text.strip().split()
    def token_to_ids(self,token):
        return[self.token_to_id.get(t, self.token_to_id["<unk>"]) for t in self._tokenized_text(token)]
    def tokenize(self,text):
        return self._tokenized_text(text)
    def vocab_size(self):
        return len(self.vocab)
    
##制作字典
def build_vocab(word_counts, min_freq=5):
    vocab = {"<pad>":0, "<unk>":1}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(word_counts, min_freq=5)
tokenizer = Tokenizer(vocab)

import torch
import torch.nn as nn
import tqdm
from torch.utils.data import Dataset
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.proscessed_data = []
        for text,label in tqdm.tqdm(zip(texts,labels),total=len(texts)):
            token_ids = self.tokenizer.token_to_ids(text)
            len_token_ids = len(token_ids)
            if len_token_ids < self.max_length:
                self.proscessed_data.append({"token_ids":token_ids,"label":label})
            else:
                stride = max_length * 0.8
                for start in range(0, len_token_ids, int(stride)):
                    end = start + max_length
                    self.proscessed_data.append({"token_ids":token_ids[start:end],"label":label})
                    if end >= len_token_ids:
                        break
    def __len__(self):
        return len(self.proscessed_data)
    def __getitem__(self, idx):
        return self.proscessed_data[idx]
    
def collate_fn(batch):
    max_len = max(len(item["token_ids"]) for item in batch)
    batch_token_ids,batch_labels = [],[]
    for item in batch:
        token_ids = item["token_ids"]
        padding_length = max_len - len(token_ids)
        batch_token_ids.append(token_ids + [tokenizer.token_to_id["<pad>"]]*padding_length)
        batch_labels.append(item["label"])
    return {
        "token_ids":torch.tensor(batch_token_ids, dtype=torch.long),
        "labels":torch.tensor(batch_labels, dtype=torch.long)
    }
    
from torch.utils.data import DataLoader
train_dataset = TextClassificationDataset(train_data_raw["data"], train_data_raw["target"], tokenizer, max_length=128)
test_dataset = TextClassificationDataset(test_data_raw["data"], test_data_raw["target"], tokenizer, max_length=128)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print("训练集样本数:", len(train_dataset))
print("测试集样本数:", len(test_dataset))
        
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.feature_extractor = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim*2),
            nn.ReLU(),
            nn.Linear(hidden_dim*2, hidden_dim*4),
            nn.ReLU()
        )
        self.classifier = nn.Linear(hidden_dim*4, num_classes)
        
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        x = self.feature_extractor(embeddings)
        mask = (input_ids != 0).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1) ##平均池化
        logits = self.classifier(x)
        return logits
    
import os
import json

class Trainer:
    def __init__(self,model,optimizer,criterion,train_loader,test_loader,device,save_path):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.save_path = save_path
        self.model.to(self.device)
        self.best_accuracy = 0.0
        os.makedirs(self.save_path, exist_ok=True)
        # 用于记录历史数据
        self.train_losses = []
        self.val_accuracies = []
    def _run_epoch(self,epoch):
        self.model.train()
        total_loss = 0.0
        for batch in tqdm.tqdm(self.train_loader,desc=f"Epoch {epoch+1}[Training]"):
            input_ids = batch["token_ids"].to(self.device)
            labels = batch["labels"].to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(input_ids)
            loss = self.criterion(logits, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(self.train_loader)
        return avg_loss
    def _evaluate(self,epoch):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in tqdm.tqdm(self.test_loader,desc=f"Epoch {epoch+1}[Evaluating]"):
                input_ids = batch["token_ids"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids)
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        accuracy = correct / total
        return accuracy
    
    def _save_checkpoint(self, epoch, val_accuracy):
        self.best_accuracy = val_accuracy
        save_path = os.path.join(self.save_path, "best_model.pth")
        torch.save(self.model.state_dict(), save_path)
        print(f"新最佳模型已保存! Epoch: {epoch+1}, 验证集准确率: {val_accuracy:.4f}")
    def train(self, num_epochs,tokenizer,label_map):
        for epoch in range(num_epochs):
            self.train_loss = []
            self.val_accuracy = []
            avg_loss = self._run_epoch(epoch)
            accuracy = self._evaluate(epoch)
            self.train_losses.append(avg_loss)
            self.val_accuracies.append(accuracy)
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Accuracy={accuracy:.4f}")
            if accuracy > self.best_accuracy:
                self.best_accuracy = accuracy
                self._save_checkpoint(epoch, accuracy)
        vocab_path = os.path.join(self.save_path, 'vocab.json')
        with open(vocab_path, 'w', encoding='utf-8') as f:
           json.dump(tokenizer.vocab, f, ensure_ascii=False, indent=4)
           
        labels_path = os.path.join(self.save_path, 'label_map.json')
        with open(labels_path, 'w', encoding='utf-8') as f:
           json.dump(label_map, f, ensure_ascii=False, indent=4)
        print(f"词典 ({vocab_path}) 和标签映射 ({labels_path}) 已保存。")
        return self.train_losses,self.val_accuracies
    
h_params = {
    "vocab_size": len(tokenizer.vocab),
    "embed_dim": 128,
    "hidden_dim": 256,
    "num_classes": len(train_data_raw["target_names"]),
    "num_epochs": 20,
    "learning_rate": 1e-3,
    "save_path": "./text_classifier_checkpoints",
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
}

model = TextClassifier(h_params["vocab_size"], h_params["embed_dim"], h_params["hidden_dim"], h_params["num_classes"]).to(h_params["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=h_params["learning_rate"])
criterion = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, criterion, train_loader, test_loader, h_params["device"], h_params["save_path"])
label_map = {idx: name for idx, name in enumerate(train_data_raw["target_names"])}
train_losses, val_accuracies = trainer.train(h_params["num_epochs"],tokenizer,label_map)

import matplotlib.pyplot as plt
        
def plot_history(train_losses, val_accuracies, title_prefix=""):
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # 绘制训练损失曲线
    ax1.plot(epochs, train_losses, 'bo-', label='Training Loss')
    ax1.set_title(f'{title_prefix} Training Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.grid(True)
    ax1.legend()
    
    # 绘制验证集准确率曲线
    ax2.plot(epochs, val_accuracies, 'ro-', label='Validation Accuracy')
    ax2.set_title(f'{title_prefix} Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)
    ax2.legend()
    
    plt.suptitle(f'{title_prefix} Training and Validation Metrics', fontsize=16)
    plt.show()

# 调用绘图函数
plot_history(train_losses, val_accuracies, title_prefix="Feed-Forward Network")           
        


100%|██████████| 1502/1502 [00:00<00:00, 6655.05it/s]


训练集样本数: 8974
测试集样本数: 6577


Epoch 1[Training]:  48%|████▊     | 135/281 [00:05<00:06, 23.71it/s]

In [ ]:
class Predictor:
    def __init__(self,model,tokenizer,label_map,device,max_length=128):
        self.model = model
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.device = device
        self.max_length = max_length
        # 修正1：从 JSON 加载的 key 是字符串，需要转为整数
        self.id_to_label = {int(idx): label for idx, label in label_map.items()}

    def predict(self,text):
        self.model.eval()
        chunks = []
        token_ids = self.tokenizer.token_to_ids(text)

        if not token_ids:
            return "无法预测（文本为空或无法分词）"

        if (len(token_ids) < self.max_length):
            # 对于短文本，直接填充到 max_length
            padding_length = self.max_length - len(token_ids)
            padded_ids = token_ids + [self.tokenizer.token_to_id["<pad>"]] * padding_length
            chunks.append(padded_ids)
        else:
            # 对于长文本，进行滑动窗口切分
            stride = int(self.max_length * 0.8)
            for start in range(0, len(token_ids), stride):
                end = start + self.max_length
                chunk = token_ids[start:end]
                
                # 修正2：对每个 chunk（特别是最后一个）进行填充
                if len(chunk) < self.max_length:
                    padding_length = self.max_length - len(chunk)
                    chunk += [self.tokenizer.token_to_id["<pad>"]] * padding_length
                
                chunks.append(chunk)
                if end >= len(token_ids):
                    break
        
        # 使用 torch.tensor()，这是推荐的创建张量的方式
        input_ids = torch.tensor(chunks, dtype=torch.long).to(self.device)
        
        with torch.no_grad():
            logits = self.model(input_ids)
            # 对所有 chunk 的预测结果进行投票
            preds = torch.argmax(logits, dim=1)
        
        # 找出票数最多的预测结果
        final_pred = torch.bincount(preds).argmax().item()
        return self.id_to_label.get(final_pred, "未知标签")
    
vocal_path = os.path.join(h_params["save_path"], 'vocab.json')
with open(vocal_path, 'r', encoding='utf-8') as f:
    saved_vocab = json.load(f)
# saved_tokenizer = Tokenizer(saved_vocab) # 这行是多余的

labels_path = os.path.join(h_params["save_path"], 'label_map.json')
with open(labels_path, 'r', encoding='utf-8') as f:
    saved_label_map = json.load(f)

inference_tokenizer = Tokenizer(vocab = saved_vocab)
inference_model = TextClassifier(
    vocab_size=len(saved_vocab),
    embed_dim=h_params["embed_dim"],
    hidden_dim=h_params["hidden_dim"],
    num_classes=len(saved_label_map)
).to(h_params["device"])
model_path = os.path.join(h_params["save_path"], "best_model.pth")
inference_model.load_state_dict(torch.load(model_path, map_location=h_params["device"]))

# 将修正后的 saved_label_map 传入
predictor = Predictor(inference_model, inference_tokenizer, saved_label_map, h_params["device"])

test_text = "The doctor prescribed a new medicine for the patient's illness, focusing on its gpu accelerated healing properties."
predicted_label = predictor.predict(test_text)
print(f"输入文本: {test_text}")
print(f"预测类别: {predicted_label}")

输入文本: The doctor prescribed a new medicine for the patient's illness, focusing on its gpu accelerated healing properties.
预测类别: sci.med


# 使用RNN，即LSTM进行训练
## 在填充后返回真实长度

In [ ]:
import os
import glob
import re
from collections import Counter
import numpy as np
def load_20newsgroups_manual(data_home, subset="train", categories=None):
    base = os.path.join(data_home, "20news-bydate", f"20news-bydate-{subset}")
    all_files = []
    all_targets = []
    target_names = categories if categories else sorted(os.listdir(base))
    
    for i, cat in enumerate(target_names):
        cat_path = os.path.join(base, cat)
        if not os.path.isdir(cat_path):
            continue
        for filepath in glob.glob(os.path.join(cat_path, "*")):
            with open(filepath, "rb") as f:
                content = f.read()
            # 尝试解码（原始数据是 latin-1 / iso-8859-1）
            try:
                text = content.decode("utf-8")
            except UnicodeDecodeError:
                text = content.decode("latin-1")
            all_files.append(text)
            all_targets.append(i)
    
    return {
        "data": all_files,
        "target": all_targets,
        "target_names": target_names
    }

# 使用
data_folder = "d:\\deeplearning\\scikit_learn_data"
categories = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

train_data_raw = load_20newsgroups_manual(data_folder, subset="train", categories=categories)
test_data_raw = load_20newsgroups_manual(data_folder, subset="test", categories=categories)


def basic_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9(),.!?\\'`]", " ", text)
    text = re.sub(r"([,.!?\\'`])", r" \\1 ", text)
    tokens = text.strip().split()
    return tokens

# 计算每篇文档的词元数量
train_text_lengths = [len(basic_tokenize(text)) for text in train_data_raw["data"]]

word_counts = Counter()
for text in train_data_raw["data"]:
    word_counts.update(basic_tokenize(text))

class Tokenizer:
    def __init__(self,vocab):
        self.vocab = vocab
        self.token_to_id = {token: idx for idx, token in enumerate(vocab)}
    
    @staticmethod
    def _tokenized_text(text):
        text = text.lower()
        text = re.sub(r"[^a-z0-9(),.!?\\'`]", " ", text)
        text = re.sub(r"([,.!?\\'`])", r" \\1 ", text)
        return text.strip().split()
    def token_to_ids(self,token):
        return[self.token_to_id.get(t, self.token_to_id["<unk>"]) for t in self._tokenized_text(token)]
    def tokenize(self,text):
        return self._tokenized_text(text)
    def vocab_size(self):
        return len(self.vocab)
    
##制作字典
def build_vocab(word_counts, min_freq=5):
    vocab = {"<pad>":0, "<unk>":1}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(word_counts, min_freq=5)
tokenizer = Tokenizer(vocab)

import torch
import torch.nn as nn
import tqdm
from torch.utils.data import Dataset
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.proscessed_data = []
        for text,label in tqdm.tqdm(zip(texts,labels),total=len(texts)):
            token_ids = self.tokenizer.token_to_ids(text)
            len_token_ids = len(token_ids)
            if len_token_ids < self.max_length:
                self.proscessed_data.append({"token_ids":token_ids,"label":label})
            else:
                stride = max_length * 0.8
                for start in range(0, len_token_ids, int(stride)):
                    end = start + max_length
                    self.proscessed_data.append({"token_ids":token_ids[start:end],"label":label})
                    if end >= len_token_ids:
                        break
    def __len__(self):
        return len(self.proscessed_data)
    def __getitem__(self, idx):
        return self.proscessed_data[idx]
    
def collate_fn(batch):
    max_len = max(len(item["token_ids"]) for item in batch)
    batch_token_ids,batch_labels,batch_length = [],[],[]
    for item in batch:
        token_ids = item["token_ids"]
        padding_length = max_len - len(token_ids)
        batch_token_ids.append(token_ids + [tokenizer.token_to_id["<pad>"]]*padding_length)
        batch_labels.append(item["label"])
        batch_length.append(len(token_ids))
    return {
        "token_ids":torch.tensor(batch_token_ids, dtype=torch.long),
        "labels":torch.tensor(batch_labels, dtype=torch.long),
        "lengths": torch.tensor(batch_length, dtype=torch.long)
    }
    
from torch.utils.data import DataLoader
train_dataset = TextClassificationDataset(train_data_raw["data"], train_data_raw["target"], tokenizer, max_length=128)
test_dataset = TextClassificationDataset(test_data_raw["data"], test_data_raw["target"], tokenizer, max_length=128)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print("训练集样本数:", len(train_dataset))
print("测试集样本数:", len(test_dataset))
        
class TextClassifierLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1, dropout=0.3, bidirectional=False):
        super(TextClassifierLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=bidirectional,
            batch_first=True
        )#(betch_size, seq_len, embed_dim) -> (batch_size, seq_len, hidden_dim*directions)
        num_directions = 2 if bidirectional else 1
        self.classifier = nn.Linear(hidden_dim*num_directions, num_classes)
        
    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids)
        packed_embeds = nn.utils.rnn.pack_padded_sequence(embeddings, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed_embeds) ## packed_output: (batch_size, seq_len, hidden_dim*directions), hidden: (num_layers*directions, batch_size, hidden_dim)
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # 双向 LSTM 拼接最后一层的正反向隐藏状态
        else:
            hidden = hidden[-1]  # 单向 LSTM 取最后一层的隐藏状态
        logits = self.classifier(hidden)
        return logits
    
import os
import json

class Trainer:
    def __init__(self,model,optimizer,criterion,train_loader,test_loader,device,save_path):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.save_path = save_path
        self.model.to(self.device)
        self.best_accuracy = 0.0
        os.makedirs(self.save_path, exist_ok=True)
        # 用于记录历史数据
        self.train_losses = []
        self.val_accuracies = []
    def _run_epoch(self,epoch):
        self.model.train()
        total_loss = 0.0
        for batch in tqdm.tqdm(self.train_loader,desc=f"Epoch {epoch+1}[Training]"):
            input_ids = batch["token_ids"].to(self.device)
            labels = batch["labels"].to(self.device)
            lengths = batch["lengths"].to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(input_ids, lengths)
            loss = self.criterion(logits, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(self.train_loader)
        return avg_loss
    def _evaluate(self,epoch):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in tqdm.tqdm(self.test_loader,desc=f"Epoch {epoch+1}[Evaluating]"):
                input_ids = batch["token_ids"].to(self.device)
                labels = batch["labels"].to(self.device)
                lengths = batch["lengths"].to(self.device)
                logits = self.model(input_ids, lengths)
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        accuracy = correct / total
        return accuracy
    
    def _save_checkpoint(self, epoch, val_accuracy):
        self.best_accuracy = val_accuracy
        save_path = os.path.join(self.save_path, "best_model.pth")
        torch.save(self.model.state_dict(), save_path)
        print(f"新最佳模型已保存! Epoch: {epoch+1}, 验证集准确率: {val_accuracy:.4f}")
    def train(self, num_epochs,tokenizer,label_map):
        for epoch in range(num_epochs):
            self.train_losses = []
            self.val_accuracies = []
            avg_loss = self._run_epoch(epoch)
            accuracy = self._evaluate(epoch)
            self.train_losses.append(avg_loss)
            self.val_accuracies.append(accuracy)
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Accuracy={accuracy:.4f}")
            if accuracy > self.best_accuracy:
                self.best_accuracy = accuracy
                self._save_checkpoint(epoch, accuracy)
        vocab_path = os.path.join(self.save_path, 'vocab.json')
        with open(vocab_path, 'w', encoding='utf-8') as f:
           json.dump(tokenizer.vocab, f, ensure_ascii=False, indent=4)
           
        labels_path = os.path.join(self.save_path, 'label_map.json')
        with open(labels_path, 'w', encoding='utf-8') as f:
           json.dump(label_map, f, ensure_ascii=False, indent=4)
        print(f"词典 ({vocab_path}) 和标签映射 ({labels_path}) 已保存。")
        return self.train_losses,self.val_accuracies
    
h_params = {
    "vocab_size": len(tokenizer.vocab),
    "embed_dim": 128,
    "hidden_dim": 256,
    "num_classes": len(train_data_raw["target_names"]),
    "num_epochs": 20,
    "learning_rate": 1e-3,
    "save_path": "./text_classifier_checkpoints_lstm",
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
    "num_layers": 1,
    "dropout": 0.3,
    "bidirectional": False
}

model = TextClassifierLSTM(h_params["vocab_size"], h_params["embed_dim"], h_params["hidden_dim"], h_params["num_classes"], h_params["num_layers"], h_params["dropout"], h_params["bidirectional"]).to(h_params["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=h_params["learning_rate"])
criterion = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, criterion, train_loader, test_loader, h_params["device"], h_params["save_path"])
label_map = {idx: name for idx, name in enumerate(train_data_raw["target_names"])}
train_losses, val_accuracies = trainer.train(h_params["num_epochs"],tokenizer,label_map)

import matplotlib.pyplot as plt
        
def plot_history(train_losses, val_accuracies, title_prefix=""):
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # 绘制训练损失曲线
    ax1.plot(epochs, train_losses, 'bo-', label='Training Loss')
    ax1.set_title(f'{title_prefix} Training Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.grid(True)
    ax1.legend()
    
    # 绘制验证集准确率曲线
    ax2.plot(epochs, val_accuracies, 'ro-', label='Validation Accuracy')
    ax2.set_title(f'{title_prefix} Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)
    ax2.legend()
    
    plt.suptitle(f'{title_prefix} Training and Validation Metrics', fontsize=16)
    plt.show()

# 调用绘图函数
plot_history(train_losses, val_accuracies, title_prefix="Feed-Forward Network")           
        


# BERT 微调
## 进行注意力掩码的生成

In [ ]:
import os
import ssl
import tqdm
import torch.nn as nn
import torch
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CURL_CA_BUNDLE"] = ""  # 禁用 cert 验证
ssl._create_default_https_context = ssl._create_unverified_context

from transformers import BertTokenizer
model_id = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
import glob
import re
from collections import Counter
import numpy as np
def load_20newsgroups_manual(data_home, subset="train", categories=None):
    base = os.path.join(data_home, "20news-bydate", f"20news-bydate-{subset}")
    all_files = []
    all_targets = []
    target_names = categories if categories else sorted(os.listdir(base))
    
    for i, cat in enumerate(target_names):
        cat_path = os.path.join(base, cat)
        if not os.path.isdir(cat_path):
            continue
        for filepath in glob.glob(os.path.join(cat_path, "*")):
            with open(filepath, "rb") as f:
                content = f.read()
            # 尝试解码（原始数据是 latin-1 / iso-8859-1）
            try:
                text = content.decode("utf-8")
            except UnicodeDecodeError:
                text = content.decode("latin-1")
            all_files.append(text)
            all_targets.append(i)
    
    return {
        "data": all_files,
        "target": all_targets,
        "target_names": target_names
    }

# 使用
data_folder = "d:\\deeplearning\\scikit_learn_data"
categories = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

train_data_raw = load_20newsgroups_manual(data_folder, subset="train", categories=categories)
test_data_raw = load_20newsgroups_manual(data_folder, subset="test", categories=categories)


def basic_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9(),.!?\\'`]", " ", text)
    text = re.sub(r"([,.!?\\'`])", r" \\1 ", text)
    tokens = text.strip().split()
    return tokens

# 计算每篇文档的词元数量
train_text_lengths = [len(basic_tokenize(text)) for text in train_data_raw["data"]]

word_counts = Counter()
for text in train_data_raw["data"]:
    word_counts.update(basic_tokenize(text))
    
##制作字典
def build_vocab(word_counts, min_freq=5):
    vocab = {"<pad>":0, "<unk>":1}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(word_counts, min_freq=5)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

from torch.utils.data import Dataset
class BertTextClassificationDataset(Dataset):
    def __init__(self,text,labels,tokenizer,max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.text = text
        self.labels = labels
        self.processed_data = []
        for i in tqdm.tqdm(range(len(self.text)), desc="Processing data"):
            encoding = self.tokenizer(self.text[i],add_special_tokens=True,truncation=False)
            input_ids = encoding["input_ids"]
            if len(input_ids) < self.max_length:
                self.processed_data.append({"input_ids":input_ids,"label":labels[i]})
            else:
                stride = int(self.max_length * 0.8)
                for start in range(0, len(input_ids), stride):
                    end = start + self.max_length
                    self.processed_data.append({"input_ids":input_ids[start:end],"label":labels[i]})
                    if end >= len(input_ids):
                        break
    def __len__(self):
        return len(self.processed_data)
    def __getitem__(self, idx):
        return self.processed_data[idx]
    
def bert_collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    batch_input_ids,attention_mask,batch_labels = [],[],[]
    for item in batch:
        input_ids = item["input_ids"]
        padding_length = max_len - len(input_ids)
        batch_input_ids.append(input_ids + [tokenizer.pad_token_id]*padding_length)
        attention_mask.append([1]*len(input_ids) + [0]*padding_length)
        batch_labels.append(item["label"])
    return {
        "input_ids":torch.tensor(batch_input_ids, dtype=torch.long),
        "attention_mask":torch.tensor(attention_mask, dtype=torch.long),
        "labels":torch.tensor(batch_labels, dtype=torch.long)
    }
    
from torch.utils.data import DataLoader
train_dataset = BertTextClassificationDataset(train_data_raw["data"], train_data_raw["target"], tokenizer, max_length=128)
test_dataset = BertTextClassificationDataset(test_data_raw["data"], test_data_raw["target"], tokenizer, max_length=128)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=bert_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=bert_collate_fn)

    

from transformers import BertModel
class BertTextClassifier(nn.Module):
    def __init__(self,model_id,num_classes,freeze_bert=False):
        super(BertTextClassifier,self).__init__()
        self.bert = BertModel.from_pretrained(model_id)
        self.classifier = nn.Linear(self.bert.config.hidden_size,num_classes)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
    def forward(self,input_ids,attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        outputs_pooled = outputs.pooler_output
        logits = self.classifier(outputs_pooled)
        return logits
    
import os
import json

class Trainer:
    def __init__(self,model,optimizer,criterion,train_loader,test_loader,device,save_path):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.save_path = save_path
        self.model.to(self.device)
        self.best_accuracy = 0.0
        os.makedirs(self.save_path, exist_ok=True)
        # 用于记录历史数据
        self.train_losses = []
        self.val_accuracies = []
    def _run_epoch(self,epoch):
        self.model.train()
        total_loss = 0.0
        for batch in tqdm.tqdm(self.train_loader,desc=f"Epoch {epoch+1}[Training]"):
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            labels = batch["labels"].to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(input_ids, attention_mask)
            loss = self.criterion(logits, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(self.train_loader)
        return avg_loss
    def _evaluate(self,epoch):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in tqdm.tqdm(self.test_loader,desc=f"Epoch {epoch+1}[Evaluating]"):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        accuracy = correct / total
        return accuracy
    
    def _save_checkpoint(self, epoch, val_accuracy):
        self.best_accuracy = val_accuracy
        save_path = os.path.join(self.save_path, "best_model")
        self.model.bert.save_pretrained(save_path)  # 使用 Hugging Face 的保存方法
        classifier_path = os.path.join(save_path, "classifier.pth")
        torch.save(self.model.classifier.state_dict(), classifier_path)  # 保存分类器权重
        print(f"新最佳模型已保存! Epoch: {epoch+1}, 验证集准确率: {val_accuracy:.4f}")
    def train(self, num_epochs,tokenizer,label_map):
        for epoch in range(num_epochs):
            self.train_losses = []
            self.val_accuracies = []
            avg_loss = self._run_epoch(epoch)
            accuracy = self._evaluate(epoch)
            self.train_losses.append(avg_loss)
            self.val_accuracies.append(accuracy)
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Accuracy={accuracy:.4f}")
            if accuracy > self.best_accuracy:
                self.best_accuracy = accuracy
                self._save_checkpoint(epoch, accuracy)
        vocab_path = os.path.join(self.save_path, 'vocab.json')
        with open(vocab_path, 'w', encoding='utf-8') as f:
           json.dump(tokenizer.vocab, f, ensure_ascii=False, indent=4)
           
        labels_path = os.path.join(self.save_path, 'label_map.json')
        with open(labels_path, 'w', encoding='utf-8') as f:
           json.dump(label_map, f, ensure_ascii=False, indent=4)
        print(f"词典 ({vocab_path}) 和标签映射 ({labels_path}) 已保存。")
        return self.train_losses,self.val_accuracies
    
h_params = {
    "model_id": model_id,
    "vocab_size": len(tokenizer.vocab),
    "embed_dim": 128,
    "hidden_dim": 256,
    "num_classes": len(train_data_raw["target_names"]),
    "num_epochs": 5,
    "learning_rate": 2e-5,
    "save_path": "./text_classifier_checkpoints_bert",
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
    "num_layers": 1,
    "dropout": 0.3,
    "bidirectional": False
}

model = BertTextClassifier(h_params["model_id"], h_params["num_classes"]).to(h_params["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=h_params["learning_rate"])
criterion = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, criterion, train_loader, test_loader, h_params["device"], h_params["save_path"])
label_map = {idx: name for idx, name in enumerate(train_data_raw["target_names"])}
train_losses, val_accuracies = trainer.train(h_params["num_epochs"],tokenizer,label_map)
import matplotlib.pyplot as plt
        
def plot_history(train_losses, val_accuracies, title_prefix=""):
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # 绘制训练损失曲线
    ax1.plot(epochs, train_losses, 'bo-', label='Training Loss')
    ax1.set_title(f'{title_prefix} Training Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.grid(True)
    ax1.legend()
    
    # 绘制验证集准确率曲线
    ax2.plot(epochs, val_accuracies, 'ro-', label='Validation Accuracy')
    ax2.set_title(f'{title_prefix} Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)
    ax2.legend()
    
    plt.suptitle(f'{title_prefix} Training and Validation Metrics', fontsize=16)
    plt.show()

# 调用绘图函数
plot_history(train_losses, val_accuracies, title_prefix="BERT Classifier")           

Epoch 1[Training]:  16%|█▌        | 61/378 [07:14<37:37,  7.12s/it]


KeyboardInterrupt: 

In [1]:
import torch
print(torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
import sys
print(sys.executable)  # 打印 Python 解释器的路径
print(torch.__version__) # 打印 torch 库的版本

2.10.0+cpu
CUDA 可用: False
d:\deeplearning\.venv\Scripts\python.exe
2.10.0+cpu
